# 02 · Class-conditioned generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/difflab/blob/main/notebooks/02_class_conditioned.ipynb)

Train a label-conditioned UNet on Fashion-MNIST and sample any class.

In [ ]:
# === difflab bootstrap — makes `import difflab` and the configs work anywhere ===
# Works on Colab/Kaggle (no checkout) and on a local clone. Three install paths,
# tried in order: already-installed -> GitHub clone -> uploaded difflab.zip.
import glob, os, pathlib, subprocess, sys

def _have_difflab():
    try:
        import difflab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False

ROOT = None
if not _have_difflab():
    # (1) If you pushed difflab to GitHub, set DIFFLAB_REPO and it clones+installs:
    REPO_URL = os.environ.get('DIFFLAB_REPO', '')  # e.g. 'https://github.com/you/difflab.git'
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/difflab'], check=False)
        ROOT = '/content/difflab'
    else:
        # (2) Else upload difflab.zip via the Files panel (left sidebar), then run this.
        hits = glob.glob('/content/difflab.zip') or glob.glob('difflab.zip')
        if not hits:
            from google.colab import files  # type: ignore
            print('Select difflab.zip to upload...')
            files.upload()
            hits = glob.glob('difflab.zip') or glob.glob('/content/difflab.zip')
        subprocess.run(['unzip', '-q', '-o', hits[0], '-d', '/content/difflab_pkg'], check=False)
        found = glob.glob('/content/difflab_pkg/**/pyproject.toml', recursive=True)
        ROOT = str(pathlib.Path(found[0]).parent) if found else '/content/difflab_pkg'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', ROOT], check=False)

# Resolve the repo root so we can find configs/ whether installed remotely or locally.
if ROOT is None:
    here = pathlib.Path.cwd()
    ROOT = str(here.parent if here.name == 'notebooks' else here)
CONFIGS = str(pathlib.Path(ROOT) / 'configs')

import difflab, torch
print('difflab', difflab.__version__, '| configs:', CONFIGS)
print('CUDA available:', torch.cuda.is_available())

> **Compute note.** The cell below runs the *smoke* config — a tiny model and 2 optimizer steps — so it finishes in seconds on CPU and proves the pipeline. For real results, switch to the full config on a GPU runtime (Runtime → Change runtime type → GPU).

In [ ]:
from difflab.config import load_config
from difflab.training import class_conditioned

cfg = load_config(f'{CONFIGS}/class_conditioned_fashionmnist_smoke.yaml')
result = class_conditioned.run(cfg)
result

## Sample one image per class (0-9)

In [ ]:
import torch
from diffusers import UNet2DModel
from difflab.config import load_config
from difflab.models import build_scheduler
from difflab.sampling import ddim_sample
from difflab.utils import get_device, make_image_grid, tensor_to_pil

cfg = load_config(f'{CONFIGS}/class_conditioned_fashionmnist_smoke.yaml')
device = get_device()
model = UNet2DModel.from_pretrained(result.checkpoint_dir).to(device)
scheduler = build_scheduler(cfg.scheduler, kind='ddim')
labels = torch.arange(8, device=device) % cfg.model.num_classes
imgs = ddim_sample(model, scheduler, num_samples=8, num_inference_steps=20,
                   class_labels=labels, device=device)
grid = make_image_grid(tensor_to_pil(imgs))
grid